# 2023 Budget Act Parser
The codes within this file enable the "scraping" and "parsing" of necessary information from SB101, which is the 2023 Budget Act.  
Note that the processing for the Trailer Bills is handled in a separate file.

In [4]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://leginfo.legislature.ca.gov/faces/billTextClient.xhtml?bill_id=202320240SB101"

# -------------------------
# Fetch and clean bill text
# -------------------------
def fetch_text(url):
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    return soup.get_text("\n")

# -------------------------
# Patterns
# -------------------------
amount_only_line_pattern = re.compile(r"^[\(\−\-]?\$?\d[\d,]*[\)]?$")

agency_header_pattern = re.compile(r"^[A-Z][A-Z0-9\/\-\s]+$")
skip_agency_headers = re.compile(r"^(CHAPTER|SECTION|ARTICLE)\b", re.IGNORECASE)

item_pattern = re.compile(r"^(\d{4}-\d{3}-\d{4})[—\-]\s*(.+)$")
object_pattern = re.compile(
    r"^(?:\([a-z\d\.]+\)\s*)?(?:Reimbursements\s+to\s+)?(\d{6,7})(?!\d)[—\-]?\s*(.+?)(?:\.{2,}|…)?$", 
    re.IGNORECASE
)
program_pattern = re.compile(
    r"^(?:\([a-z\d\.]+\)\s*)?(?:Reimbursements\s+to\s+)?(\d{4})(?!\d)[—\-]?\s*(.+?)(?:\.{2,}|…)?$", 
    re.IGNORECASE
)

schedule_header_pattern = re.compile(r"^Schedule:", re.IGNORECASE)
schedule_group_pattern = re.compile(r"^\(([\d\.]+)\)\s*$")

dotleader_trailing_amount = re.compile(
    r"(?:\.{2,}|…)\s*\$?([\d]{1,3}(?:,\d{3})+|\d+)\s*$"
)

# -------------------------
# Helpers
# -------------------------
def clean_amount(s: str) -> int:
    s = s.replace(",", "").replace("$", "").replace("−", "-").replace("—", "-").strip()
    
    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1].strip()
        
    if not s or (len(s) <= 2 and s != "0"):
        return None 
        
    try:
        return int(float(s))
    except:
        return None

def extract_dotleader_amount(line: str):
    m = dotleader_trailing_amount.search(line)
    if m:
        return int(m.group(1).replace(",", ""))
    return None

def find_amount_for_line(lines, i, max_lookahead=6):
    amt = extract_dotleader_amount(lines[i])
    if amt is not None:
        return amt, i + 1

    for j in range(i + 1, min(i + 1 + max_lookahead, len(lines))):
        line_to_check = lines[j]
        if schedule_group_pattern.match(line_to_check) or amount_only_line_pattern.match(line_to_check):
            potential_amt = clean_amount(line_to_check)
            if potential_amt is not None:
                return potential_amt, j + 1

    for j in range(i + 1, min(i + 1 + max_lookahead, len(lines))):
        amt2 = extract_dotleader_amount(lines[j])
        if amt2 is not None:
            return amt2, j + 1

    return None, i + 1

def extract_fund_name_from_item_title(item_title: str):
    m = re.search(r"payable\s+from\s+the\s+(.+)$", item_title, flags=re.IGNORECASE)
    return m.group(1).strip().rstrip(".") if m else None

# -------------------------
# Parser Logic
# -------------------------
def parse_budget_money_only(text):
    start_match = re.search(r"SEC\.\s+2\.00", text)
    if start_match:
        text = text[start_match.start():]
    
    end_match = re.search(r"SEC\.\s+3\.00", text)
    if end_match:
        text = text[:end_match.start()]

    lines = [l.strip() for l in text.splitlines() if l.strip()]

    current_agency = None
    current_item = None
    current_fund_code = None
    current_fund_name = None

    in_schedule = False
    current_schedule_group = None

    rows = []
    i = 0

    while i < len(lines):
        line = lines[i]
        line_s = line.strip()

        # 1. Finish if we reach the next Section
        if re.match(r"^SEC\.\s+3\.00", line_s):
            break

        # 2. Skip lines that are just amounts (likely totals or subtotals)
        m_special = re.match(r"^(\d{4}-\d{3,4})(?!\d)", line_s)
        if m_special and not item_pattern.match(line_s):
            current_item = None
            in_schedule = False
            current_schedule_group = None
            i += 1
            while i < len(lines):
                if item_pattern.match(lines[i].strip()):
                    break
                i += 1
            continue

        # 3. Judge if it's an Agency header (all caps, not too short, and not a known non-agency header)
        if (
            agency_header_pattern.match(line)
            and len(line) > 8
            and not skip_agency_headers.match(line)
        ):
            current_agency = line_s
            i += 1
            continue

        # 4. Judge if it's an Item line
        m_item = item_pattern.match(line)
        if m_item:
            current_item = m_item.group(1)
            item_title = m_item.group(2).strip()
            
            in_schedule = False
            current_schedule_group = None
            current_fund_code = current_item.split("-")[-1]
            current_fund_name = extract_fund_name_from_item_title(item_title)

            amt, next_i = find_amount_for_line(lines, i)
            rows.append({
                "level": "item", "agency": current_agency, "item_number": current_item,
                "program_code": None, "object_code": None, "fund_code": current_fund_code,
                "fund_name": current_fund_name, "schedule_group": None, "name": item_title, "amount": amt or 0
            })
            i = next_i
            continue

        # 5. Judge if it's a Schedule header or group
        if schedule_header_pattern.match(line):
            in_schedule = True
            current_schedule_group = None
            i += 1
            continue

        m_sg = schedule_group_pattern.match(line)
        if m_sg and in_schedule:
            current_schedule_group = m_sg.group(1)
            i += 1
            continue

        if line_s.lower().startswith("provisions:"):
            i += 1
            while i < len(lines):
                nl = lines[i].strip()
                if item_pattern.match(nl) or nl.startswith(("SEC.", "SECTION")): break
                i += 1
            continue

        # 6. Object and Program lines should only be processed if we have a current Item context
        if current_item is None:
            i += 1
            continue

        m_obj = re.match(r"^(?:\([a-z\d\.]+\)\s*)?(?:Reimbursements\s+to\s+)?(\d{6,7})(?!\d)[—\-\s]+(.+)", line, re.IGNORECASE)
        if m_obj:
            obj_code = m_obj.group(1)
            if not line_s.startswith("Up to") and len(obj_code) >= 5:
                amt, next_i = find_amount_for_line(lines, i)
                name = re.sub(r"[\.\s…]{2,}.*$", "", m_obj.group(2).strip())
                name = re.sub(r"\s*\(?\$?[\d,]+\)?\s*$", "", name)
                is_reimb = "reimbursement" in line.lower()
                rows.append({
                    "level": "object", "agency": current_agency, "item_number": current_item,
                    "program_code": None, "object_code": obj_code, "fund_code": current_fund_code,
                    "fund_name": current_fund_name, "schedule_group": current_schedule_group,
                    "name": name.strip(), "amount": -abs(amt) if (amt and is_reimb) else (amt or 0)
                })
                i = next_i
                continue

        m_prog = re.match(r"^(?:\([a-z\d\.]+\)\s*)?(?:Reimbursements\s+to\s+)?(\d{4})(?!\d)[—\-\s]+(.+)", line, re.IGNORECASE)
        if m_prog:
            prog_code = m_prog.group(1)
            if not line_s.startswith("Up to") and len(prog_code) >= 3:
                amt, next_i = find_amount_for_line(lines, i)
                name = re.sub(r"[\.\s…]{2,}.*$", "", m_prog.group(2).strip())
                name = re.sub(r"\s*\(?\$?[\d,]+\)?\s*$", "", name)
                rows.append({
                    "level": "program", "agency": current_agency, "item_number": current_item,
                    "program_code": prog_code, "object_code": None, "fund_code": current_fund_code,
                    "fund_name": current_fund_name, "schedule_group": current_schedule_group,
                    "name": name.strip(), "amount": amt or 0
                })
                i = next_i
                continue

        i += 1

    return rows

# -------------------------
# Run + save CSV
# -------------------------
def main():
    print("Fetching and Parsing...")
    text = fetch_text(URL)
    data = parse_budget_money_only(text)
    df_raw = pd.DataFrame(data)

    print("Cleaning up duplicate total rows...")
    
    item_totals = df_raw[df_raw["level"] == "item"].set_index("item_number")["amount"].to_dict()

    def remove_duplicate_totals(row):
        if row["level"] == "program":
            item_no = row["item_number"]
            all_in_item = df_raw[df_raw["item_number"] == item_no]
            objects_in_item = all_in_item[all_in_item["level"] == "object"]
            objects_in_group = all_in_item[
                (all_in_item["level"] == "object") & 
                (all_in_item["schedule_group"] == row["schedule_group"])
            ]

            if not objects_in_group.empty:
                group_obj_sum = objects_in_group["amount"].sum()
                if abs(row["amount"]) == abs(group_obj_sum) and row["amount"] != 0:
                    return False 

            if objects_in_group.empty:
                return True

        return True

    df_final = df_raw[df_raw.apply(remove_duplicate_totals, axis=1)].copy()

    # save CSV
    out = "SB101_2023_cleaned.csv"
    df_final.to_csv(out, index=False, encoding='utf-8-sig')
    print(f"File successfully saved to: {out}")
    print(f"Final Row Count: {len(df_final)}")

if __name__ == "__main__":
    main()

Fetching and Parsing...
Cleaning up duplicate total rows...
File successfully saved to: SB101_2023_cleaned.csv
Final Row Count: 4482


In [5]:
df = pd.read_csv(
    "SB101_2023_cleaned.csv",  
    dtype={"program_code": "string", "object_code": "string", "fund_code": "string", "schedule_group": "string"}
)
df.head(200)

,level,agency,item_number,program_code,object_code,fund_code,fund_name,schedule_group,name,amount
0,item,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,<NA>,<NA>,0001,NaN,<NA>,For support of Senate,177325000
1,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,<NA>,101001,0001,NaN,1,Salaries of Senators,6751000
2,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,<NA>,317295,0001,NaN,1,Mileage,11000
3,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,<NA>,317292,0001,NaN,1,Expenses,1712000
4,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,<NA>,500004,0001,NaN,1,Operating Expenses,168851000
...,...,...,...,...,...,...,...,...,...,...
195,item,LEGISLATIVE/JUDICIAL/EXECUTIVE,0521-001-0890,<NA>,<NA>,0890,Federal Trust Fund,<NA>,"For support of Secretary of Transportation, pa...",7882000
196,program,LEGISLATIVE/JUDICIAL/EXECUTIVE,0521-001-0890,0275,<NA>,0890,Federal Trust Fund,1,California Traffic Safety Program,7882000
197,item,LEGISLATIVE/JUDICIAL/EXECUTIVE,0521-001-3228,<NA>,<NA>,3228,Greenhouse Gas Reduction Fund,<NA>,"For support of Secretary of Transportation, pa...",74000
198,program,LEGISLATIVE/JUDICIAL/EXECUTIVE,0521-001-3228,0276,<NA>,3228,Greenhouse Gas Reduction Fund,1,Transit and Intercity Rail Capital,74000
